# Contrato municipal comum — SIDRA e SINAN

A chave de integração `codigo_municipio` é texto com seis dígitos. Códigos de sete dígitos perdem apenas o dígito verificador final; valores inválidos viram nulos. O código original permanece em `codigo_municipio_original`.

O nome legível preserva acentos e maiúsculas/minúsculas da fonte, removendo o sufixo de UF e espaços excedentes. `municipio_original` preserva o valor recebido. **Joins usam código, nunca nome.**

In [0]:
from pyspark.sql import functions as F


def padronizar_municipios(df, coluna_codigo="municipio_codigo", coluna_nome="municipio"):
    obrigatorias = {coluna_codigo}
    if coluna_nome:
        obrigatorias.add(coluna_nome)
    ausentes = obrigatorias - set(df.columns)
    if ausentes:
        raise ValueError(f"Colunas municipais ausentes: {sorted(ausentes)}")

    codigo = F.trim(F.col(coluna_codigo).cast("string"))
    resultado = (
        df.withColumn("codigo_municipio_original", F.col(coluna_codigo))
        .withColumn(
            "codigo_municipio",
            F.when(codigo.rlike(r"^\d{7}$"), F.substring(codigo, 1, 6))
            .when(codigo.rlike(r"^\d{6}$"), codigo),
        )
    )
    if coluna_nome:
        nome = F.regexp_replace(
            F.regexp_replace(F.trim(F.col(coluna_nome)), r"\s+\([A-Z]{2}\)$", ""),
            r"\s+", " ",
        )
        resultado = (
            resultado.withColumn("municipio_original", F.col(coluna_nome))
            .withColumn("municipio_padronizado", nome)
        )
    return resultado